# Notebook 05 — Méthode Hybride & Évaluation Comparative

**Objectif** : Combiner le modèle fine-tuné (LoRA) avec la récupération FAISS pour former une méthode hybride, puis évaluer et comparer les 4 méthodes selon 6 axes.

| # | Méthode       | Description |
|---|---------------|-------------|
| 1 | **Baseline**  | LLaMA 3.1 8B via Groq — sans contexte |
| 2 | **RAG**       | LLaMA 3.1 8B via Groq + top-5 chunks FAISS (docs bruts) |
| 3 | **Fine-tuné** | LLaMA 3.1 8B + LoRA (fine-tuning sur train.json) |
| 4 | **Hybride**   | LLaMA 3.1 8B + LoRA + top-5 chunks FAISS |

**Métriques** : Exact Match, F1 Token, BERTScore, ROUGE-L, Taux d'hallucination, Latence

**Corpus FAISS** : documents bruts chunkés (400 mots, overlap 100) — `wikipedia_technique.json` + `arxiv.json` + `lemonde.json`

**9 figures** :
1. Métriques globales (EM / F1 / BERTScore / ROUGE-L) par méthode
2. BERTScore par strate temporelle × méthode
3. Performance par type de question × méthode
4. Latence moyenne par méthode
5. Taux d'hallucination estimé par méthode
6. Trade-off qualité vs latence (scatter)
7. Heatmap BERTScore méthodes × dataset_type
8. Comparaison questions simples vs complexes (Arxiv)
9. **Accuracy par seuil BERTScore** — % réponses "correctes" à seuil ≥80% / ≥85% / ≥90%

> ⚠️ **GPU OBLIGATOIRE** — Aller dans `Exécution > Modifier le type d'exécution > GPU`

**Inputs** :
- `BASE_PATH/models/lora_adapter/`, `BASE_PATH/models/faiss_index/`
- `BASE_PATH/data/processed/test.json`
- `BASE_PATH/results/baseline_predictions.json`, `rag_predictions.json`, `finetuned_predictions.json`

**Outputs** :
- `BASE_PATH/results/hybrid_predictions.json`, `final_report.json`
- `BASE_PATH/results/plots/` — 6 graphiques PNG

## 0. Vérification GPU

In [ ]:
# Vérification que le GPU est bien disponible avant de continuer
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "Aucun GPU détecté !\n"
        "→ Allez dans Exécution > Modifier le type d'exécution > GPU, puis relancez."
    )

gpu_name   = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU détecté  : {gpu_name}")
print(f"VRAM totale  : {gpu_mem_gb:.1f} Go")
print(f"CUDA version : {torch.version.cuda}")

## 1. Montage Google Drive

In [ ]:
# Montage du Drive et définition du chemin de base du projet
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/llm-integration-study/'

## 2. Installation des dépendances

In [ ]:
# Installation d'Unsloth pour charger l'adaptateur LoRA
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
# Installation des bibliothèques d'évaluation et de visualisation
!pip install -q faiss-cpu sentence-transformers bert-score rouge-score matplotlib transformers accelerate peft bitsandbytes

## 3. Imports et configuration

In [ ]:
# Imports de toutes les bibliothèques nécessaires
import os, json, time, string
import numpy as np
import pandas as pd
import faiss
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter, defaultdict
from bert_score import score as bert_score_fn
from rouge_score import rouge_scorer as _rouge_scorer
from sentence_transformers import SentenceTransformer
from unsloth import FastLanguageModel
from tqdm.notebook import tqdm
import torch

# Chemins Drive
PROCESSED_PATH = os.path.join(BASE_PATH, 'data', 'processed')
MODELS_PATH    = os.path.join(BASE_PATH, 'models', 'lora_adapter')
FAISS_PATH     = os.path.join(BASE_PATH, 'models', 'faiss_index')
RESULTS_PATH   = os.path.join(BASE_PATH, 'results')
PLOTS_PATH     = os.path.join(BASE_PATH, 'results', 'plots')

for path in [RESULTS_PATH, PLOTS_PATH]:
    os.makedirs(path, exist_ok=True)

# LLaMA 3.1 8B — même modèle que Baseline/RAG → comparaison équitable des 4 méthodes
BASE_MODEL  = "unsloth/Meta-Llama-3.1-8B-bnb-4bit"
EMBED_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"  # multilingue FR/EN
TOP_K       = 5
MAX_SEQ_LEN = 2048

# Scorer ROUGE (instancié une seule fois)
_ROUGE = _rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)

METHODS_ORDER    = ["Baseline", "RAG", "Fine-tuné", "Hybride"]
STRATA_ORDER     = ["récent", "intermédiaire", "fondamental"]
QTYPES_ORDER     = ["factuel", "synthese", "comprehension"]
METHOD_COLORS    = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']
STRATA_COLORS    = ['#e74c3c', '#f39c12', '#2ecc71']
QTYPE_COLORS     = ['#3498db', '#9b59b6', '#1abc9c']

# Seuils BERTScore pour le calcul d'accuracy (valeurs en [0,1])
BS_THRESHOLDS = [0.80, 0.85, 0.90]  # → 80%, 85%, 90%
BS_THRESHOLD  = 0.85                 # seuil principal affiché dans les tableaux

print("Configuration chargée.")
print(f"  Seuils accuracy : {[int(t*100) for t in BS_THRESHOLDS]}%  (principal : {int(BS_THRESHOLD*100)}%)")
print(f"  Modèle LoRA  : {MODELS_PATH}")
print(f"  Index FAISS  : {FAISS_PATH}")
print(f"  Résultats    : {RESULTS_PATH}")

## 4. Chargement des données et résultats précédents

In [ ]:
# Chargement de tous les fichiers JSON depuis Drive
def load_json(path, label=""):
    """Charge un fichier JSON avec gestion d'erreur."""
    try:
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f"  [OK] {label or path} ({len(data)} entrées)")
        return data
    except FileNotFoundError:
        print(f"  [MANQUANT] {path}")
        return []
    except json.JSONDecodeError as e:
        print(f"  [ERROR JSON] {path} : {e}")
        return []

print("Chargement des fichiers...")
test_data             = load_json(os.path.join(PROCESSED_PATH,  'test.json'),                   'test.json')
baseline_predictions  = load_json(os.path.join(RESULTS_PATH,    'baseline_predictions.json'),   'baseline_predictions.json')
rag_predictions       = load_json(os.path.join(RESULTS_PATH,    'rag_predictions.json'),        'rag_predictions.json')
finetuned_predictions = load_json(os.path.join(RESULTS_PATH,    'finetuned_predictions.json'),  'finetuned_predictions.json')
corpus_meta           = load_json(os.path.join(FAISS_PATH,      'metadata.json'),               'faiss metadata.json')

print(f"\nTest set : {len(test_data)} questions")

## 5. Chargement du modèle fine-tuné + index FAISS

In [ ]:
# Chargement du modèle de base + adaptateur LoRA depuis Drive
print(f"Chargement du modèle de base : {BASE_MODEL}")
print("(Téléchargement si nécessaire, peut prendre 5-10 min)")

try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=MAX_SEQ_LEN,
        dtype=None,
        load_in_4bit=True,
    )
    print("Modèle de base chargé.")
except Exception as e:
    raise RuntimeError(f"Échec chargement modèle : {e}")

# Chargement de l'adaptateur LoRA
try:
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, MODELS_PATH)
    print(f"Adaptateur LoRA chargé depuis : {MODELS_PATH}")
except Exception as e:
    raise RuntimeError(f"Échec chargement LoRA : {e}\n→ Assurez-vous d'avoir exécuté 04_finetuning.ipynb.")

FastLanguageModel.for_inference(model)
print("Modèle en mode inférence.")

In [ ]:
# Chargement de l'index FAISS et du modèle d'embedding
faiss_index_file = os.path.join(FAISS_PATH, 'index.faiss')

try:
    index = faiss.read_index(faiss_index_file)
    print(f"Index FAISS chargé : {index.ntotal} vecteurs, dim {index.d}")
except Exception as e:
    raise RuntimeError(f"Échec chargement FAISS : {e}\n→ Assurez-vous d'avoir exécuté 03_baseline_rag.ipynb.")

print(f"Chargement du modèle d'embedding : {EMBED_MODEL}")
embed_model = SentenceTransformer(EMBED_MODEL)
print("Modèle d'embedding chargé.")

## 6. HYBRIDE — Inférence LoRA + FAISS

In [ ]:
# Fonctions de récupération FAISS et de génération hybride
HYBRID_TEMPLATE = (
    "### Instruction: Réponds à cette question en te basant sur le contexte fourni.\n"
    "### Context: {context_block}\n"
    "### Input: {question}\n"
    "### Response:"
)
_STOP_MARKERS = ["\n### Instruction:", "\n### Input:", "\n### Context:", "\n### Response:"]

def retrieve_top_k(question, k=TOP_K):
    q_emb = embed_model.encode(
        [question], convert_to_numpy=True, normalize_embeddings=True
    ).astype(np.float32)
    scores, indices = index.search(q_emb, k)
    chunks = [corpus_meta[i] for i in indices[0] if i < len(corpus_meta)]
    return chunks, scores[0].tolist()

def generate_hybrid(question, max_new_tokens=200):
    """Récupère les top-K chunks puis génère avec le modèle fine-tuné."""
    try:
        chunks, scores = retrieve_top_k(question, k=TOP_K)
        context_block  = "\n\n".join([f"[{c.get('title','')[:50]}] {c.get('text', c.get('context',''))[:400]}" for c in chunks])
        prompt = HYBRID_TEMPLATE.format(context_block=context_block, question=question)

        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        start  = time.time()
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.eos_token_id,
            )
        latency_ms = round((time.time() - start) * 1000)

        prompt_len = inputs["input_ids"].shape[1]
        answer = tokenizer.decode(outputs[0][prompt_len:], skip_special_tokens=True).strip()
        for marker in _STOP_MARKERS:
            if marker in answer:
                answer = answer.split(marker)[0].strip()

        return answer, latency_ms, [f"{c.get('doc_id',c.get('pair_id',''))}#{c.get('chunk_idx','')}" for c in chunks], scores
    except Exception as e:
        print(f"  [ERROR] generate_hybrid : {e}")
        return "", 0, [], []

print("Fonctions hybrides prêtes.")

In [ ]:
# Exécution de la méthode hybride sur tout le test set
hybrid_predictions = []

for item in tqdm(test_data, desc="Hybride (LoRA + FAISS)"):
    question    = item.get('question', '')
    true_answer = item.get('answer', '')

    predicted, latency, chunk_ids, scores = generate_hybrid(question)

    hybrid_predictions.append({
        "pair_id":          item.get('pair_id', ''),
        "question":         question,
        "predicted_answer": predicted,
        "true_answer":      true_answer,
        "latency_ms":       latency,
        "retrieved_chunks": chunk_ids,
        "retrieval_scores": scores,
        "method":           "hybrid"
    })

latencies = [p['latency_ms'] for p in hybrid_predictions if p['latency_ms'] > 0]
print(f"\nHybride terminé : {len(hybrid_predictions)} prédictions")
print(f"Latence moyenne  : {np.mean(latencies):.0f} ms" if latencies else "Latence : N/A")

In [ ]:
# Sauvegarde des prédictions hybrides sur Drive
hybrid_path = os.path.join(RESULTS_PATH, 'hybrid_predictions.json')
try:
    with open(hybrid_path, 'w', encoding='utf-8') as f:
        json.dump(hybrid_predictions, f, ensure_ascii=False, indent=2)
    print(f"Hybride sauvegardé : {hybrid_path}  ({os.path.getsize(hybrid_path)/1024:.1f} Ko)")
except Exception as e:
    print(f"[ERROR] Sauvegarde hybride : {e}")

## 7. Évaluation des 4 méthodes

In [ ]:
# Métriques d'évaluation : EM, F1, BERTScore, ROUGE-L, hallucination
def normalize_text(text):
    text = text.lower().strip()
    text = text.translate(str.maketrans('', '', string.punctuation))
    stop = {'a','an','the','le','la','les','un','une','des'}
    return ' '.join(t for t in text.split() if t not in stop)

def exact_match(pred, gold):
    return int(normalize_text(pred) == normalize_text(gold))

def f1_token(pred, gold):
    pred_tok = normalize_text(pred).split()
    gold_tok = normalize_text(gold).split()
    if not pred_tok or not gold_tok:
        return 0.0
    common = Counter(pred_tok) & Counter(gold_tok)
    n = sum(common.values())
    if n == 0:
        return 0.0
    p = n / len(pred_tok)
    r = n / len(gold_tok)
    return 2 * p * r / (p + r)

def rouge_l(pred, gold):
    if not pred.strip() or not gold.strip():
        return 0.0
    return _ROUGE.score(gold, pred)['rougeL'].fmeasure

def compute_bert_score(preds, refs, batch_size=32):
    try:
        _, _, F = bert_score_fn(preds, refs, lang="fr",
                                model_type="distilbert-base-multilingual-cased",
                                batch_size=batch_size, verbose=False)
        return F.tolist()
    except Exception as e:
        print(f"  [ERROR] BERTScore : {e}")
        return [0.0] * len(preds)

def hallucination_score(pred, context):
    """Proxy d'hallucination : faible overlap avec le contexte source → hallucination probable."""
    if not pred.strip() or not context.strip():
        return 1.0   # prédit vide = hallucine tout
    return 1.0 - rouge_l(pred, context)

def evaluate_method(predictions, method_name, test_lookup=None):
    """Calcule toutes les métriques pour une liste de prédictions.
    test_lookup : dict pair_id → item test (pour recency + question_type)
    """
    if not predictions:
        print(f"  [WARN] Aucune prédiction pour '{method_name}'")
        base = {"method": method_name, "n": 0}
        for k in ["exact_match","f1","bertscore","rouge_l","hallucination","latency_ms"]:
            base[k] = 0.0
        return base

    em_list, f1_list, rl_list, hall_list, lat_list = [], [], [], [], []
    preds_list, refs_list = [], []

    # Résultats détaillés par strate et type
    by_recency = defaultdict(lambda: {"preds": [], "refs": [], "hall": []})
    by_qtype   = defaultdict(lambda: {"preds": [], "refs": [], "hall": []})
    by_dstype  = defaultdict(lambda: {"preds": [], "refs": [], "hall": []})

    for p in predictions:
        pred    = p.get('predicted_answer', '') or ''
        gold    = p.get('true_answer', '')      or ''
        pair_id = p.get('pair_id', '')

        # Récupérer les métadonnées depuis test.json
        recency = 'inconnu'
        qtype   = 'inconnu'
        context = ''
        if test_lookup and pair_id in test_lookup:
            item    = test_lookup[pair_id]
            recency = item.get('recency_category', 'inconnu')
            qtype   = item.get('question_type', 'inconnu')
            context = item.get('context', '')

        em_list.append(exact_match(pred, gold))
        f1_list.append(f1_token(pred, gold))
        rl_list.append(rouge_l(pred, gold))
        hall_list.append(hallucination_score(pred, context if context else gold))

        preds_list.append(pred if pred else " ")
        refs_list.append(gold  if gold  else " ")

        by_recency[recency]["preds"].append(preds_list[-1])
        by_recency[recency]["refs"].append(refs_list[-1])
        by_recency[recency]["hall"].append(hall_list[-1])
        by_qtype[qtype]["preds"].append(preds_list[-1])
        by_qtype[qtype]["refs"].append(refs_list[-1])
        by_qtype[qtype]["hall"].append(hall_list[-1])
        dstype = item.get("dataset_type", "inconnu") if test_lookup and pair_id in test_lookup else "inconnu"
        by_dstype[dstype]["preds"].append(preds_list[-1])
        by_dstype[dstype]["refs"].append(refs_list[-1])
        by_dstype[dstype]["hall"].append(hall_list[-1])

        if p.get('latency_ms', 0) > 0:
            lat_list.append(p['latency_ms'])

    print(f"  Calcul BERTScore pour '{method_name}' ({len(preds_list)} paires)...")
    bs_list = compute_bert_score(preds_list, refs_list)

    # BERTScore par strate
    bs_by_recency = {}
    for cat, d in by_recency.items():
        scores = compute_bert_score(d["preds"], d["refs"]) if d["preds"] else [0.0]
        bs_by_recency[cat] = round(np.mean(scores) * 100, 2)

    bs_by_qtype = {}
    for qt, d in by_qtype.items():
        scores = compute_bert_score(d["preds"], d["refs"]) if d["preds"] else [0.0]
        bs_by_qtype[qt] = round(np.mean(scores) * 100, 2)

    hall_by_recency = {k: round(np.mean(v["hall"]) * 100, 2) for k, v in by_recency.items()}
    hall_by_qtype   = {k: round(np.mean(v["hall"]) * 100, 2) for k, v in by_qtype.items()}

    # Accuracy par seuil BERTScore
    accuracy_by_threshold = {
        f"acc_bs{int(t*100)}": round(sum(s >= t for s in bs_list) / len(bs_list) * 100, 2)
        for t in BS_THRESHOLDS
    }

    return {
        "method":          method_name,
        "exact_match":     round(np.mean(em_list)   * 100, 2),
        "f1":              round(np.mean(f1_list)   * 100, 2),
        "bertscore":       round(np.mean(bs_list)   * 100, 2),
        "rouge_l":         round(np.mean(rl_list)   * 100, 2),
        "hallucination":   round(np.mean(hall_list) * 100, 2),
        "latency_ms":      round(np.mean(lat_list), 1) if lat_list else 0,
        "n":               len(predictions),
        **accuracy_by_threshold,
        "bs_by_recency":   bs_by_recency,
        "bs_by_qtype":     bs_by_qtype,
        "hall_by_recency": hall_by_recency,
        "hall_by_qtype":   hall_by_qtype,
        "bs_by_dstype":    {k: round(np.mean(compute_bert_score(v["preds"], v["refs"])) * 100, 2) if v["preds"] else 0.0 for k, v in by_dstype.items()},
        "hall_by_dstype":  {k: round(np.mean(v["hall"]) * 100, 2) for k, v in by_dstype.items()},
    }

print("Fonctions d'évaluation prêtes (EM / F1 / BERTScore / ROUGE-L / Hallucination / Accuracy@BS).")

In [ ]:
# Calcul des métriques pour les 4 méthodes
# Construire un lookup pair_id → item test pour récupérer recency + question_type
test_lookup = {item.get('pair_id', ''): item for item in test_data}

print("Évaluation en cours...\n")
all_results = []
for method_name, preds in [
    ("Baseline",  baseline_predictions),
    ("RAG",       rag_predictions),
    ("Fine-tuné", finetuned_predictions),
    ("Hybride",   hybrid_predictions),
]:
    result = evaluate_method(preds, method_name, test_lookup=test_lookup)
    all_results.append(result)
    acc85 = result.get('acc_bs85', 0)
    print(f"  {method_name:<12} EM={result['exact_match']:5.1f}%  F1={result['f1']:5.1f}%  "
          f"BERTScore={result['bertscore']:5.1f}%  ROUGE-L={result['rouge_l']:5.1f}%  "
          f"Acc@85%={acc85:5.1f}%  Halluc.={result['hallucination']:5.1f}%  Latence={result['latency_ms']:.0f}ms")

print("\nÉvaluation terminée.")
print(f"\n--- Accuracy par seuil BERTScore ---")
print(f"  {'Méthode':<12}  {'≥80%':>8}  {'≥85%':>8}  {'≥90%':>8}")
print("  " + "-" * 40)
for r in all_results:
    print(f"  {r['method']:<12}  {r.get('acc_bs80',0):>7.1f}%  {r.get('acc_bs85',0):>7.1f}%  {r.get('acc_bs90',0):>7.1f}%")

In [ ]:
# Tableau comparatif pandas
df = pd.DataFrame([{
    "Méthode":           r['method'],
    "Exact Match (%)":   r['exact_match'],
    "F1 Token (%)":      r['f1'],
    "BERTScore (%)":     r['bertscore'],
    "ROUGE-L (%)":       r['rouge_l'],
    "Acc@85% (%)":       r.get('acc_bs85', 0),
    "Hallucination (%)": r['hallucination'],
    "Latence moy. (ms)": r['latency_ms'],
    "N":                 r['n'],
} for r in all_results]).set_index("Méthode")

print("=" * 80)
print("TABLEAU COMPARATIF — 4 méthodes")
print("=" * 80)
display(df.style
    .highlight_max(subset=["Exact Match (%)","F1 Token (%)","BERTScore (%)","ROUGE-L (%)","Acc@85% (%)"], color='lightgreen')
    .highlight_min(subset=["Hallucination (%)","Latence moy. (ms)"], color='lightblue')
    .format(precision=1)
)

print(f"\n  Acc@85% = % de réponses avec BERTScore ≥ 85%  (seuil de 'réponse correcte')")
print(f"  Interprétation : une réponse est 'correcte' si sa similarité sémantique")
print(f"  avec la référence dépasse le seuil (plus souple qu'Exact Match)")

In [ ]:
# Sauvegarde du rapport final en JSON
report = {
    "generated_at": time.strftime('%Y-%m-%dT%H:%M:%S'),
    "n_test_samples": len(test_data),
    "bs_thresholds_used": [int(t * 100) for t in BS_THRESHOLDS],
    "methods": all_results
}
report_path = os.path.join(RESULTS_PATH, 'final_report.json')

try:
    with open(report_path, 'w', encoding='utf-8') as f:
        json.dump(report, f, ensure_ascii=False, indent=2)
    print(f"Rapport sauvegardé : {report_path}  ({os.path.getsize(report_path)/1024:.1f} Ko)")
except Exception as e:
    print(f"[ERROR] Sauvegarde rapport : {e}")

## 8. Visualisations

In [ ]:
# ── Figure 1 : Métriques globales (EM / F1 / BERTScore / ROUGE-L) ──────────────
methods  = [r['method']    for r in all_results]
metrics  = {
    "Exact Match (%)": [r['exact_match'] for r in all_results],
    "F1 Token (%)":    [r['f1']          for r in all_results],
    "BERTScore (%)":   [r['bertscore']   for r in all_results],
    "ROUGE-L (%)":     [r['rouge_l']     for r in all_results],
}
colors4  = ['#4C72B0','#55A868','#C44E52','#8172B2']

x     = np.arange(len(methods))
width = 0.20
fig1, ax1 = plt.subplots(figsize=(12, 6))
for i, (label, vals) in enumerate(metrics.items()):
    offset = (i - 1.5) * width
    bars = ax1.bar(x + offset, vals, width, label=label, color=colors4[i], alpha=0.87)
    for bar in bars:
        h = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2, h + 0.5, f'{h:.1f}',
                 ha='center', va='bottom', fontsize=7.5)

ax1.set_xlabel('Méthode', fontsize=12)
ax1.set_ylabel('Score (%)', fontsize=12)
ax1.set_title('Figure 1 — Métriques globales par méthode\n(Exact Match, F1 Token, BERTScore, ROUGE-L)',
              fontsize=13, fontweight='bold')
ax1.set_xticks(x); ax1.set_xticklabels(methods, fontsize=11)
ax1.set_ylim(0, 115); ax1.legend(fontsize=9)
ax1.grid(axis='y', alpha=0.3)
ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)
plt.tight_layout()
p1 = os.path.join(PLOTS_PATH, 'fig1_global_metrics.png')
fig1.savefig(p1, dpi=150, bbox_inches='tight'); plt.show()
print(f"Figure 1 sauvegardée : {p1}")

In [ ]:
# ── Figure 2 : BERTScore par strate temporelle × méthode ───────────────────────
fig2, ax2 = plt.subplots(figsize=(11, 6))
x2    = np.arange(len(METHODS_ORDER))
w2    = 0.22
for i, (cat, col) in enumerate(zip(STRATA_ORDER, STRATA_COLORS)):
    vals = [r['bs_by_recency'].get(cat, 0) for r in all_results]
    offset = (i - 1) * w2
    bars = ax2.bar(x2 + offset, vals, w2, label=cat.capitalize(), color=col, alpha=0.87)
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax2.text(bar.get_x() + bar.get_width()/2, h + 0.4, f'{h:.1f}',
                     ha='center', va='bottom', fontsize=7.5)

ax2.set_xlabel('Méthode', fontsize=12)
ax2.set_ylabel('BERTScore (%)', fontsize=12)
ax2.set_title('Figure 2 — BERTScore par strate temporelle × méthode\n'
              '(récent 2022-2025 / intermédiaire 2019-2022 / fondamental pré-2019)',
              fontsize=12, fontweight='bold')
ax2.set_xticks(x2); ax2.set_xticklabels(METHODS_ORDER, fontsize=11)
ax2.set_ylim(0, 115); ax2.legend(title='Strate', fontsize=9)
ax2.grid(axis='y', alpha=0.3)
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)
plt.tight_layout()
p2 = os.path.join(PLOTS_PATH, 'fig2_bertscore_by_recency.png')
fig2.savefig(p2, dpi=150, bbox_inches='tight'); plt.show()
print(f"Figure 2 sauvegardée : {p2}")

In [ ]:
# ── Figure 3 : Performance par type de question × méthode ──────────────────────
fig3, ax3 = plt.subplots(figsize=(11, 6))
x3 = np.arange(len(METHODS_ORDER))
w3 = 0.22
for i, (qt, col) in enumerate(zip(QTYPES_ORDER, QTYPE_COLORS)):
    vals = [r['bs_by_qtype'].get(qt, 0) for r in all_results]
    offset = (i - 1) * w3
    bars = ax3.bar(x3 + offset, vals, w3, label=qt.capitalize(), color=col, alpha=0.87)
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax3.text(bar.get_x() + bar.get_width()/2, h + 0.4, f'{h:.1f}',
                     ha='center', va='bottom', fontsize=7.5)

ax3.set_xlabel('Méthode', fontsize=12)
ax3.set_ylabel('BERTScore (%)', fontsize=12)
ax3.set_title('Figure 3 — BERTScore par type de question × méthode\n'
              '(factuel / synthèse / compréhension)',
              fontsize=12, fontweight='bold')
ax3.set_xticks(x3); ax3.set_xticklabels(METHODS_ORDER, fontsize=11)
ax3.set_ylim(0, 115); ax3.legend(title='Type de question', fontsize=9)
ax3.grid(axis='y', alpha=0.3)
ax3.spines['top'].set_visible(False); ax3.spines['right'].set_visible(False)
plt.tight_layout()
p3 = os.path.join(PLOTS_PATH, 'fig3_bertscore_by_qtype.png')
fig3.savefig(p3, dpi=150, bbox_inches='tight'); plt.show()
print(f"Figure 3 sauvegardée : {p3}")

In [ ]:
# ── Figure 4 : Latence moyenne par méthode ─────────────────────────────────────
lat_vals = [r['latency_ms'] for r in all_results]
fig4, ax4 = plt.subplots(figsize=(8, 5))
bars4 = ax4.bar(METHODS_ORDER, lat_vals, color=METHOD_COLORS, alpha=0.87, width=0.5)
for bar, val in zip(bars4, lat_vals):
    ax4.text(bar.get_x() + bar.get_width()/2, val + max(lat_vals)*0.01,
             f'{val:,.0f} ms', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax4.set_xlabel('Méthode', fontsize=12)
ax4.set_ylabel('Latence moyenne (ms)', fontsize=12)
ax4.set_title('Figure 4 — Latence moyenne par méthode', fontsize=13, fontweight='bold')
ax4.set_ylim(0, max(lat_vals) * 1.25 if lat_vals else 1)
ax4.grid(axis='y', alpha=0.3)
ax4.spines['top'].set_visible(False); ax4.spines['right'].set_visible(False)
plt.tight_layout()
p4 = os.path.join(PLOTS_PATH, 'fig4_latency.png')
fig4.savefig(p4, dpi=150, bbox_inches='tight'); plt.show()
print(f"Figure 4 sauvegardée : {p4}")

In [ ]:
# ── Figure 5 : Taux d'hallucination estimé par méthode ─────────────────────────
hall_vals = [r['hallucination'] for r in all_results]
fig5, axes = plt.subplots(1, 2, figsize=(14, 5))

# 5a — Hallucination globale
bars5 = axes[0].bar(METHODS_ORDER, hall_vals, color=METHOD_COLORS, alpha=0.87, width=0.5)
for bar, val in zip(bars5, hall_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.5,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_xlabel('Méthode', fontsize=12)
axes[0].set_ylabel('Taux d\'hallucination estimé (%)', fontsize=11)
axes[0].set_title('5a — Global', fontsize=11, fontweight='bold')
axes[0].set_ylim(0, max(hall_vals) * 1.35 if hall_vals else 100)
axes[0].grid(axis='y', alpha=0.3)
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

# 5b — Hallucination par strate temporelle
x5b = np.arange(len(STRATA_ORDER))
w5b = 0.18
for i, (method, col) in enumerate(zip(METHODS_ORDER, METHOD_COLORS)):
    r = next((r for r in all_results if r['method'] == method), {})
    vals5b = [r.get('hall_by_recency', {}).get(cat, 0) for cat in STRATA_ORDER]
    axes[1].bar(x5b + (i - 1.5) * w5b, vals5b, w5b, label=method, color=col, alpha=0.87)
axes[1].set_xlabel('Strate temporelle', fontsize=12)
axes[1].set_ylabel('Taux d\'hallucination (%)', fontsize=11)
axes[1].set_title('5b — Par strate temporelle', fontsize=11, fontweight='bold')
axes[1].set_xticks(x5b)
axes[1].set_xticklabels([s.capitalize() for s in STRATA_ORDER], fontsize=10)
axes[1].set_ylim(0, 110); axes[1].legend(fontsize=8)
axes[1].grid(axis='y', alpha=0.3)
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

fig5.suptitle('Figure 5 — Taux d\'hallucination estimé\n(proxy : 1 − ROUGE-L(prédit, contexte source))',
              fontsize=12, fontweight='bold')
plt.tight_layout()
p5 = os.path.join(PLOTS_PATH, 'fig5_hallucination.png')
fig5.savefig(p5, dpi=150, bbox_inches='tight'); plt.show()
print(f"Figure 5 sauvegardée : {p5}")

In [ ]:
# ── Figure 6 : Trade-off Qualité vs Latence (scatter) ──────────────────────────
fig6, ax6 = plt.subplots(figsize=(8, 6))
for r, col in zip(all_results, METHOD_COLORS):
    ax6.scatter(r['latency_ms'], r['bertscore'],
                s=220, color=col, zorder=5, edgecolors='white', linewidths=1.5)
    ax6.annotate(r['method'],
                 xy=(r['latency_ms'], r['bertscore']),
                 xytext=(8, 4), textcoords='offset points',
                 fontsize=11, fontweight='bold', color=col)

ax6.set_xlabel('Latence moyenne (ms)  ← plus rapide', fontsize=12)
ax6.set_ylabel('BERTScore (%)  ↑ meilleure qualité', fontsize=12)
ax6.set_title('Figure 6 — Trade-off Qualité vs Latence\n(idéal : coin haut-gauche)',
              fontsize=12, fontweight='bold')
ax6.grid(alpha=0.3)
ax6.spines['top'].set_visible(False); ax6.spines['right'].set_visible(False)

# Zone idéale
ymin, ymax = ax6.get_ylim()
xmin, xmax = ax6.get_xlim()
ax6.annotate('← Idéal (rapide + précis)', xy=(xmin, ymax),
             xytext=(xmin + (xmax-xmin)*0.02, ymax - (ymax-ymin)*0.05),
             fontsize=9, color='green', style='italic')

plt.tight_layout()
p6 = os.path.join(PLOTS_PATH, 'fig6_quality_vs_latency.png')
fig6.savefig(p6, dpi=150, bbox_inches='tight'); plt.show()
print(f"Figure 6 sauvegardée : {p6}")

## 8b. Tableau croisé méthodes × dataset_type

In [ ]:
# ── Tableau croisé : BERTScore par méthode × dataset_type ────────────────────
DATASET_TYPES = ["technique", "multisauts", "temporel"]

crosstable_rows = []
for r in all_results:
    row = {"Méthode": r["method"]}
    for dt in DATASET_TYPES:
        row[dt.capitalize()] = r.get("bs_by_dstype", {}).get(dt, 0.0)
    row["Moyenne"] = r["bertscore"]
    crosstable_rows.append(row)

df_cross = pd.DataFrame(crosstable_rows).set_index("Méthode")

print("\nBERTScore (%) par méthode × dataset_type")
print("=" * 60)
print(df_cross.to_string())

# Highlight best per column
def highlight_max_col(col):
    return ['font-weight: bold; color: green' if v == col.max() else '' for v in col]

print("\nArxiv — comparaison simple vs complexe :")
for r in all_results:
    bs = r.get("bs_by_qtype", {})
    s  = bs.get("simple", 0)
    c  = bs.get("complexe", 0)
    print(f"  {r['method']:<12}  simple={s:.1f}%  complexe={c:.1f}%  delta={c-s:+.1f}%")

In [ ]:
# ── Figure 9 : Accuracy par seuil BERTScore ──────────────────────────────────
threshold_labels = [f"≥{int(t*100)}%" for t in BS_THRESHOLDS]
threshold_keys   = [f"acc_bs{int(t*100)}" for t in BS_THRESHOLDS]

fig9, axes9 = plt.subplots(1, 2, figsize=(14, 5))

# 9a — Barres groupées : accuracy à 3 seuils pour les 4 méthodes
x9    = np.arange(len(METHODS_ORDER))
w9    = 0.22
shade_colors = ['#2ecc71', '#f39c12', '#e74c3c']  # vert, orange, rouge selon seuil
for i, (key, label, col) in enumerate(zip(threshold_keys, threshold_labels, shade_colors)):
    vals = [r.get(key, 0) for r in all_results]
    offset = (i - 1) * w9
    bars = axes9[0].bar(x9 + offset, vals, w9, label=label, color=col, alpha=0.80)
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            axes9[0].text(bar.get_x() + bar.get_width()/2, h + 0.8,
                          f'{h:.0f}', ha='center', va='bottom', fontsize=8)

axes9[0].set_xlabel('Méthode', fontsize=12)
axes9[0].set_ylabel('Accuracy (%)', fontsize=12)
axes9[0].set_title('9a — % réponses "correctes" selon le seuil BERTScore', fontsize=11, fontweight='bold')
axes9[0].set_xticks(x9); axes9[0].set_xticklabels(METHODS_ORDER, fontsize=11)
axes9[0].set_ylim(0, 115); axes9[0].legend(title='Seuil BERTScore', fontsize=9)
axes9[0].grid(axis='y', alpha=0.3)
axes9[0].spines['top'].set_visible(False); axes9[0].spines['right'].set_visible(False)

# 9b — Courbe d'accuracy vs seuil (de 0.70 à 0.95) pour chaque méthode
# On ne peut calculer ça que si on a accès aux scores individuels — ici on affiche les 3 seuils calculés
thresholds_pct = [int(t * 100) for t in BS_THRESHOLDS]
for r, col in zip(all_results, METHOD_COLORS):
    acc_vals = [r.get(f"acc_bs{t}", 0) for t in thresholds_pct]
    axes9[1].plot(thresholds_pct, acc_vals, marker='o', color=col,
                  linewidth=2, markersize=8, label=r['method'])
    for x_pt, y_pt in zip(thresholds_pct, acc_vals):
        axes9[1].annotate(f'{y_pt:.0f}%', xy=(x_pt, y_pt),
                          xytext=(3, 5), textcoords='offset points', fontsize=8, color=col)

axes9[1].set_xlabel('Seuil BERTScore (%)', fontsize=12)
axes9[1].set_ylabel('% réponses au-dessus du seuil', fontsize=12)
axes9[1].set_title('9b — Courbe accuracy vs seuil par méthode', fontsize=11, fontweight='bold')
axes9[1].set_xticks(thresholds_pct)
axes9[1].set_xticklabels([f'{t}%' for t in thresholds_pct])
axes9[1].set_ylim(0, 115); axes9[1].legend(fontsize=9)
axes9[1].grid(alpha=0.3)
axes9[1].spines['top'].set_visible(False); axes9[1].spines['right'].set_visible(False)

fig9.suptitle('Figure 9 — Pourcentage de réponses "correctes" par seuil BERTScore\n'
              '(Acc@X% = % réponses avec BERTScore ≥ X% — plus souple qu\'Exact Match)',
              fontsize=12, fontweight='bold')
plt.tight_layout()
p9 = os.path.join(PLOTS_PATH, 'fig9_accuracy_by_threshold.png')
fig9.savefig(p9, dpi=150, bbox_inches='tight'); plt.show()
print(f"Figure 9 sauvegardée : {p9}")

# Afficher le résumé textuel
print(f"\n--- Résumé Accuracy@{int(BS_THRESHOLD*100)}% ---")
for r in all_results:
    acc = r.get(f'acc_bs{int(BS_THRESHOLD*100)}', 0)
    print(f"  {r['method']:<12} : {acc:.1f}% des réponses ont BERTScore ≥ {int(BS_THRESHOLD*100)}%")

In [ ]:
# ── Figure 7 : Heatmap BERTScore méthodes × dataset_type ─────────────────────
import numpy as np

heatmap_data = df_cross[[dt.capitalize() for dt in DATASET_TYPES]].values.astype(float)

fig7, ax7 = plt.subplots(figsize=(8, 5))
im = ax7.imshow(heatmap_data, cmap="YlOrRd", aspect="auto",
                vmin=max(0, heatmap_data.min() - 5),
                vmax=min(100, heatmap_data.max() + 5))

ax7.set_xticks(range(len(DATASET_TYPES)))
ax7.set_xticklabels([dt.capitalize() for dt in DATASET_TYPES], fontsize=12)
ax7.set_yticks(range(len(METHODS_ORDER)))
ax7.set_yticklabels(METHODS_ORDER, fontsize=12)

for i in range(len(METHODS_ORDER)):
    for j in range(len(DATASET_TYPES)):
        val = heatmap_data[i, j]
        text_color = "white" if val < (heatmap_data.min() + heatmap_data.max()) / 2 else "black"
        ax7.text(j, i, f"{val:.1f}%", ha="center", va="center",
                 fontsize=11, fontweight="bold", color=text_color)

plt.colorbar(im, ax=ax7, label="BERTScore (%)")
ax7.set_title("Figure 7 — BERTScore par méthode × dataset_type\n"
              "(technique=Wikipedia FR | multisauts=Arxiv | temporel=Le Monde)",
              fontsize=12, fontweight="bold")
fig7.tight_layout()
p7 = os.path.join(PLOTS_PATH, "fig7_heatmap_methodes_datasets.png")
fig7.savefig(p7, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure 7 sauvegardée : {p7}")

# ── Figure 8 : Arxiv simple vs complexe (bar chart) ──────────────────────────
fig8, ax8 = plt.subplots(figsize=(8, 5))
x8 = np.arange(len(METHODS_ORDER))
w8 = 0.35
simple_vals  = [r.get("bs_by_qtype", {}).get("simple",   0) for r in all_results]
complex_vals = [r.get("bs_by_qtype", {}).get("complexe", 0) for r in all_results]

bars_s = ax8.bar(x8 - w8/2, simple_vals,  w8, label="Simple (1 fait)",           color="#4C72B0", alpha=0.85)
bars_c = ax8.bar(x8 + w8/2, complex_vals, w8, label="Complexe (multi-sauts)", color="#DD8452", alpha=0.85)
for bars in [bars_s, bars_c]:
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax8.text(bar.get_x() + bar.get_width()/2, h + 0.4, f"{h:.1f}",
                     ha="center", va="bottom", fontsize=8)

ax8.set_xticks(x8); ax8.set_xticklabels(METHODS_ORDER, fontsize=11)
ax8.set_ylabel("BERTScore (%)", fontsize=12)
ax8.set_title("Figure 8 — Questions simples vs complexes sur Arxiv\n"
              "(les questions multi-sauts sont-elles plus difficiles ?)",
              fontsize=12, fontweight="bold")
ax8.legend(fontsize=10); ax8.set_ylim(0, 115)
ax8.grid(axis="y", alpha=0.3)
ax8.spines["top"].set_visible(False); ax8.spines["right"].set_visible(False)
fig8.tight_layout()
p8 = os.path.join(PLOTS_PATH, "fig8_simple_vs_complexe.png")
fig8.savefig(p8, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure 8 sauvegardée : {p8}")

## 9. Résumé final du projet complet

In [ ]:
# Récapitulatif complet de tous les fichiers produits sur Drive
print("=" * 70)
print("RÉSUMÉ FINAL — Pipeline complet 'Intégration de nouvelles informations dans les LLMs'")
print("=" * 70)

all_output_files = [
    ("01_scraping",        os.path.join(BASE_PATH, 'data', 'raw', 'wikipedia.json')),
    ("01_scraping",        os.path.join(BASE_PATH, 'data', 'raw', 'arxiv.json')),
    ("02_dataset_builder", os.path.join(BASE_PATH, 'data', 'processed', 'train.json')),
    ("02_dataset_builder", os.path.join(BASE_PATH, 'data', 'processed', 'test.json')),
    ("03_baseline_rag",    os.path.join(BASE_PATH, 'results', 'baseline_predictions.json')),
    ("03_baseline_rag",    os.path.join(BASE_PATH, 'results', 'rag_predictions.json')),
    ("03_baseline_rag",    os.path.join(FAISS_PATH, 'index.faiss')),
    ("04_finetuning",      os.path.join(BASE_PATH, 'results', 'finetuned_predictions.json')),
    ("05_hybrid_eval",     os.path.join(BASE_PATH, 'results', 'hybrid_predictions.json')),
    ("05_hybrid_eval",     report_path),
] + [("05_hybrid_eval", p) for p in [p1, p2, p3, p4, p5, p6]]

print(f"\n{'Notebook':<22} {'Taille':>8}   Fichier")
print("-" * 70)
for nb, fpath in all_output_files:
    try:
        size = f"{os.path.getsize(fpath)/1024:>6.1f} Ko"
    except Exception:
        size = "  N/A   "
    print(f"{nb:<22} {size}   {fpath}")

# Meilleures performances par métrique
print("\n--- Meilleures performances ---")
best_em   = max(all_results, key=lambda x: x['exact_match'])
best_f1   = max(all_results, key=lambda x: x['f1'])
best_bs   = max(all_results, key=lambda x: x['bertscore'])
best_rl   = max(all_results, key=lambda x: x['rouge_l'])
best_hall = min(all_results, key=lambda x: x['hallucination'])
best_lt   = min([r for r in all_results if r['latency_ms'] > 0], key=lambda x: x['latency_ms'], default=all_results[0])

print(f"  Exact Match      : {best_em['method']:<12} {best_em['exact_match']:.1f}%")
print(f"  F1 Token         : {best_f1['method']:<12} {best_f1['f1']:.1f}%")
print(f"  BERTScore        : {best_bs['method']:<12} {best_bs['bertscore']:.1f}%")
print(f"  ROUGE-L          : {best_rl['method']:<12} {best_rl['rouge_l']:.1f}%")
print(f"  Hallucination \u2193  : {best_hall['method']:<12} {best_hall['hallucination']:.1f}%")
print(f"  Latence min      : {best_lt['method']:<12} {best_lt['latency_ms']:.0f} ms")

print("\n✔ Pipeline complet terminé. Tous les résultats sont sur Google Drive.")
print(f"  → {BASE_PATH}")
print("=" * 70)